# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.

R/= Taxi-v4 tiene 500 estados y 6 acciones.

El número de estados sale de combinar todas las variables que describen la situación: 25 posiciones posibles del taxi (grid de 5x5), 5 ubicaciones del pasajero (4 puntos de recogida + estar dentro del taxi) y 4 destinos posibles → 25 × 5 × 4 = 500.

FrozenLake solo necesita representar la posición del agente en la grilla (16 estados en la versión 4x4), mientras que Taxi debe codificar además dónde está el pasajero y a dónde quiere ir. Al combinar varias variables independientes el espacio de estados crece de forma multiplicativa, por eso Taxi tiene muchos más estados que FrozenLake aunque el mapa sea de tamaño parecido.


## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [ ]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?

r/= En la interacción del bloque anterior:

$s_t$ → state: el estado inicial del ambiente, obtenido con env.reset(seed=42). Es la codificación numérica de la posición del taxi, la ubicación del pasajero y el destino en ese momento.
$a_t$ → action: la acción elegida (en este caso al azar, con env.action_space.sample()), un número entre 0 y 5 que representa moverse en alguna dirección, recoger o dejar al pasajero.
$r_{t+1}$ → reward: la recompensa que entrega el ambiente como consecuencia de haber ejecutado esa acción desde state. Refleja qué tan buena o mala fue la decisión (por ejemplo, penaliza movimientos innecesarios o un dropoff/pickup incorrecto).
$s_{t+1}$ → next_state: el nuevo estado al que queda el ambiente después de aplicar la acción, es decir, cómo cambió la posición del taxi (y posiblemente la del pasajero) tras el movimiento.

En resumen, la tupla $(s_t, a_t, r{t+1}, s{t+1})$ describe una transición completa: "estando en el estado $s_t$, tomé la acción $a_t$, recibí la recompensa $r{t+1}$ y terminé en el estado $s{t+1}$", que es justamente la información que usa Q-Learning para actualizar la tabla.


## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$

R/= El agente debe aprender un valor $Q(s,a)$ para cada combinación posible de estado y acción:

$$
|\mathcal{S}| \times |\mathcal{A}| = 500 \times 6 = 3000
$$

Es decir, la Q-table tiene 3000 valores en total que el agente debe ir ajustando durante el entrenamiento.

## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [ ]:
def choose_action(Q, state, epsilon, env):
    # con probabilidad epsilon exploramos con una accion al azar
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    # si no, explotamos la mejor accion conocida para ese estado
    return int(np.argmax(Q[state]))



## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [ ]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    # mejor valor Q posible desde el siguiente estado
    best_next = np.max(Q[next_state])

    # valor objetivo segun la formula de Q-learning
    target = reward + gamma * best_next

    # error TD: diferencia entre el objetivo y el valor actual
    td_error = target - Q[state, action]

    # actualizamos Q(s,a) moviendolo en direccion del error
    Q[state, action] += alpha * td_error

## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):

            # escoger accion con la politica epsilon-greedy
            action = choose_action(Q, state, epsilon, env)

            # ejecutar la accion en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)

            # actualizar la Q-table con la transicion observada
            update_q(Q, state, action, reward, next_state, alpha, gamma)

            # avanzar al siguiente estado y acumular recompensa
            state = next_state
            total_reward += reward

            # terminar el episodio si corresponde
            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards

## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.3
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?

R/=
- Sí, la recompensa promedio mejora de forma muy marcada. Al inicio del entrenamiento el promedio arranca en valores muy negativos, cercanos a -320, porque el agente actúa casi al azar y comete muchos errores (choques, pickups/dropoffs inválidos, exceso de pasos).
- La curva sube de forma pronunciada durante los primeros 1000-1200 episodios, que es el tramo donde el agente aprende la mayor parte de la política. A partir de ahí la mejora se hace mucho más lenta y desde aproximadamente el episodio 1500 en adelante la curva se estabiliza, oscilando cerca de 0 a 5 de recompensa promedio hasta el final del entrenamiento (episodio 5000).
- Este comportamiento no indica una convergencia perfecta, sino que el agente alcanzó una política razonablemente buena y estable. Las pequeñas oscilaciones que se ven incluso al final (entre 0 y 5) se deben a que sigue existiendo exploración ($\epsilon = 0.1$), lo que ocasionalmente lleva al agente a tomar acciones subóptimas.

## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
- ¿El taxi sin entrenamiento logra completar la tarea?
- ¿El agente entrenado evita acciones innecesarias?
- ¿Qué evidencia visual le permite afirmar que el agente aprendió?

R/=
- El taxi sin entrenamiento se mueve prácticamente al azar entre las 6 acciones posibles (ya que con la Q-table en ceros todas las acciones tienen el mismo valor). Se le ve deambular por el mapa, chocar contra las paredes, e intentar recoger o dejar al pasajero en lugares incorrectos.
- No, el taxi sin entrenamiento normalmente no logra completar la tarea dentro del límite de pasos (50); se queda dando vueltas sin lógica y termina por truncamiento, no porque haya entregado al pasajero.
- El agente entrenado sí evita acciones innecesarias: va directo hacia el pasajero, lo recoge en el momento correcto, se dirige al destino por el camino más corto y hace el dropoff sin desperdiciar movimientos.
- La evidencia visual más clara es la eficiencia de la ruta: el agente entrenado completa el episodio en pocos pasos y con una recompensa total positiva, mientras que el no entrenado se mueve sin rumbo y termina con recompensa muy negativa (o sin completar el viaje). Esa diferencia de comportamiento es la prueba de que la Q-table aprendió una política útil.

## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?

R/=
- La acción con mayor valor Q es la 3 (West), con Q(123,3) = 3.949. Es claramente la mejor opción para ese estado, superando por bastante al resto (la siguiente más alta es la acción 2 con -0.151).
- Que una acción tenga un valor Q mayor que otra significa que el agente espera obtener más recompensa acumulada a largo plazo tomando esa acción desde ese estado. Aquí se ve bien: moverse al Oeste (3.949) es mucho mejor que ir al Norte (-2.330) o hacer Dropoff (-5.893), que son claramente malas decisiones en ese estado.
- No podemos interpretar Q(s,a) solo como recompensa inmediata porque incluye también el valor descontado de las recompensas futuras (gamma * max Q(s',a')). Por eso acciones como Pickup (4) o Dropoff (5) tienen valores muy negativos (-5.36 y -5.89): no es que la acción en sí cueste tanto, sino que ejecutarla en este estado lleva a una situación de la cual es difícil recuperarse (un dropoff/pickup inválido), afectando el retorno esperado de todo lo que viene después.


## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.

R/=
- Velocidad de aprendizaje: la subida inicial es parecida en forma pero arranca desde un punto peor (cerca de -380 en vez de -320), porque con epsilon = 0.3 el agente explora tres veces más seguido, así que comete más acciones al azar incluso cuando ya tiene una Q-table decente. La curva se estabiliza mas o menos en el mismo rango de episodios (1500-2000).

- Estabilidad: se nota mas ruido en la parte plana de la curva comparado con epsilon = 0.1, que se veia mas lisa. Esto es esperable porque un 30% de las acciones durante el entrenamiento siguen siendo aleatorias, lo que mete variabilidad constante en la recompensa de cada episodio.

- Recompensa final: es mas baja que antes, se estabiliza alrededor de -10 a -20, mientras que con epsilon = 0.1 se estabilizaba cerca de 0 a 5. No es que el agente aprendio peor la politica, sino que sigue "pagando" el costo de explorar tanto durante el entrenamiento.

- Comportamiento observado: el aumento de epsilon prioriza la exploracion sobre la explotacion, lo cual sirve para conocer mas estados pero sacrifica la recompensa acumulada durante el entrenamiento. Si se evaluara el agente de forma greedy (epsilon = 0), probablemente el desempeño final seria similar al de epsilon = 0.1, ya que la Q-table aprendida es parecida.


## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
